# Sentiment Classification Project

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Verify Setup
Make sure cuda (GPU) is available

# Load data

In [11]:
train_full = pd.read_csv("data/train.csv")

# Build Validation Set
Todo: 

Currently we use 90% of the reviews for training, and the remaining 10% for validation
This should be optimized. Use k-fold validation or something like that to get most of our data as training and enable proper hyperparameter tuning later on.

In [12]:
train_df, val_df = train_test_split(
        train_full, test_size=0.1, stratify=train_full["label"], random_state=42
)

# TRAIN PIPELINE

## Overview
A modular training pipeline that evaluates combinations of (vectorizer, classifier) pairs.
Defined across three files: embeddings.py, models.py, and train_loop.py.

## Step 1 — Vectorization (embeddings.py)
For classical vectorizers (BoW, TF-IDF), the signature is:
    (X_train, X_val) -> (X_train_embedded, X_val_embedded)
The vocabulary is fit on X_train only, then applied to both splits.

## Step 2 — Classification (models.py)
Each classifier is a factory function that returns a fresh, untrained model instance.
Keeping them as functions (rather than instances) ensures each combination in the
loop starts from a clean state.

model is then used to fit X_train on Y_train: model.fit(X_train, Y_train)

## Step 3 — Train Loop (train_loop.py)
Takes a list of (vectorizer_fn, model_fn) tuples. For each combination it:
  1. Vectorizes the data
  2. Trains the classifier on the training embeddings X_train, Y_train
  3. Evaluates on both train and validation splits

Returns a list of result dicts, one per combination:
    {
        "vectorizer":          str,   # name of the vectorizer function
        "model":               str,   # name of the model factory
        "training_score":      float,
        "training_mae":        float,
        "training_accuracy":   float,
        "validation_score":    float,
        "validation_mae":      float,
        "validation_accuracy": float
    }

In [15]:
from embeddings import *
from models import *
from train_loop_caching import train_loop
from gemma_embeddings import *


In [17]:
# combinations_adv = [
#     (get_gemma_embeddings, get_random_forest_v2),
#     (get_gemma_embeddings_v2,          get_random_forest_v2),
#     (get_gemma_embeddings_seq128,          get_random_forest_v2),
#     (get_gemma_embeddings_v2,          get_logistic_regression),
#     (get_gemma_embeddings_v2,          get_linear_svm),
#     (get_gemma_embeddings_v2,          get_mlp),
#     (get_gemma_embeddings_v2,          get_knn),
#     (get_gemma_embeddings_seq128,          get_logistic_regression),
#     (get_gemma_embeddings_seq128,          get_linear_svm),
#     (get_gemma_embeddings_seq128,          get_mlp),
#     (get_gemma_embeddings_seq128,          get_knn),
# ]
# results_local = train_loop(train_df, val_df, combinations_adv)
# pd.DataFrame(results_local).sort_values("validation_score", ascending=False)

In [ ]:
# from models_additional import (
#     # Expected-value decoded variants
#     get_logistic_regression_ev, get_linear_svm_ev, get_mlp_ev, get_xgboost_ev,
#     # Regression-based (direct MAE optimization)
#     get_ridge_regression, get_mlp_regressor, get_xgboost_mae,
# )

# combinations_adv = [
#     # --- Expected-value decoding variants of models you've already run ---
#     (get_gemma_embeddings_seq128, get_logistic_regression_ev),
#     (get_gemma_embeddings_seq128, get_linear_svm_ev),
#     (get_gemma_embeddings_seq128, get_mlp_ev),
#     #(get_gemma_embeddings_seq128, get_xgboost_ev),

#     # --- Regression-based (direct MAE optimization) ---
#     (get_gemma_embeddings_seq128, get_ridge_regression),
#     (get_gemma_embeddings_seq128, get_mlp_regressor),
#     #(get_gemma_embeddings_seq128, get_xgboost_mae),
# ]

# results = train_loop(train_df, val_df, combinations_adv)

# pd.DataFrame(results).sort_values("validation_score", ascending=False)

# Gemma Embeddings — Baseline Classifier Results

Embeddings model: EmbeddingGemma-300m (`google/embeddinggemma-300m`)

We ran the same classifier suite on three variants of the Gemma embedding pipeline. All three use the same underlying model; they differ in loading path, task prompt, normalization, and max sequence length.

## Variants

| Variant | Loading path | Prompt | Normalization | max_seq_length |
|---|---|---|---|---|
| `get_gemma_embeddings` (default) | `AutoModel` + manual mean pool | none | no | 128 |
| `get_gemma_embeddings_v2` | `SentenceTransformer` | Classification | yes (L2) | 64 |
| `get_gemma_embeddings_seq128` | `SentenceTransformer` | Classification | yes (L2) | 128 |

## Hyperparameters

| Classifier | Hyperparameters |
|---|---|
| LogisticRegression | `C=1.0, max_iter=1000` |
| LinearSVC | `C=1.0, max_iter=1000` |
| KNeighborsClassifier | `n_neighbors=15, n_jobs=-1` |
| MLPClassifier | `hidden_layer_sizes=(256,), max_iter=300, random_state=1` |
| MLPClassifier-v2 | `hidden_layer_sizes=(128,), alpha=1e-2, early_stopping=True, validation_fraction=0.1, n_iter_no_change=10, max_iter=300, random_state=1` |
| RandomForestClassifier | `n_estimators=300, random_state=1` |
| RandomForestClassifier-v2 | `n_estimators=300, max_depth=10, min_samples_leaf=5, random_state=1` |
| XGBoostClassifier | `n_estimators=300, max_depth=6, learning_rate=0.1` |
| XGBoostRegressor (MAE, retuned) | `n_estimators=1500, max_depth=6, learning_rate=0.1, objective="reg:absoluteerror"` |
| Ridge | `alpha=1.0` (wrapped as classifier via round+clip) |

## Results — `get_gemma_embeddings` (default: plain loading, no prompt, 128 tok)

| Classifier | Train Score | Train MAE | Train Acc | **Val Score** | **Val MAE** | **Val Acc** |
|---|---|---|---|---|---|---|
| MLPClassifier-v2 | 0.9008 | 0.3968 | 0.6547 | **0.8950** ⬅️ | **0.4201** | **0.6342** ⬅️ |
| LogisticRegression | 0.8936 | 0.4256 | 0.6321 | **0.8917** | **0.4331** | **0.6256** |
| RandomForestClassifier | 1.0000 | 0.0000 | 1.0000 | **0.8866** | **0.4537** | **0.6062** |
| LinearSVC | 0.8877 | 0.4493 | 0.6225 | **0.8852** | **0.4593** | **0.6145** |
| KNeighborsClassifier | 0.8917 | 0.4334 | 0.6425 | **0.8747** | **0.5012** | **0.5796** |
| MLPClassifier (unregularized) | 0.9947 | 0.0214 | 0.9819 | **0.8593** | **0.5628** | **0.5435** |

## Results — `get_gemma_embeddings_v2` (Classification prompt, normalized, 64 tok)

| Classifier | Train Score | Train MAE | Train Acc | **Val Score** | **Val MAE** | **Val Acc** |
|---|---|---|---|---|---|---|
| MLPClassifier-v2 | 0.8961 | 0.4155 | 0.6393 | **0.8913** ⬅️ | **0.4348** | **0.6229** ⬅️ |
| LogisticRegression | 0.8915 | 0.4340 | 0.6264 | **0.8890** | **0.4439** | **0.6168** |
| LinearSVC | 0.8873 | 0.4509 | 0.6198 | **0.8857** | **0.4574** | **0.6129** |
| RandomForestClassifier-v2 | 0.8971 | 0.4116 | 0.6470 | **0.8852** | **0.4590** | **0.6026** |
| KNeighborsClassifier | 0.8954 | 0.4182 | 0.6468 | **0.8813** | **0.4747** | **0.5937** |

## Results — `get_gemma_embeddings_seq128` (Classification prompt, normalized, 128 tok)

### Argmax decoding

| Classifier | Train Score | Train MAE | Train Acc | **Val Score** | **Val MAE** | **Val Acc** |
|---|---|---|---|---|---|---|
| MLPClassifier-v2 | 0.9010 | 0.3959 | 0.6515 | **0.8974** | **0.4104** | **0.6393** |
| XGBoostClassifier | 0.9334 | 0.2662 | 0.7734 | **0.8949** | **0.4203** | **0.6322** |
| LogisticRegression | 0.8965 | 0.4140 | 0.6395 | **0.8948** | **0.4210** | **0.6325** |
| LinearSVC | 0.8924 | 0.4304 | 0.6327 | **0.8911** | **0.4357** | **0.6270** |
| RandomForestClassifier-v2 | 0.9013 | 0.3947 | 0.6577 | **0.8903** | **0.4388** | **0.6161** |
| KNeighborsClassifier | 0.8997 | 0.4011 | 0.6576 | **0.8866** | **0.4536** | **0.6063** |

### Expected-value decoding (same models, different inference)

| Classifier | Train Score | Train MAE | Train Acc | **Val Score** | **Val MAE** | **Val Acc** |
|---|---|---|---|---|---|---|
| MLPClassifier-v2 + EV | 0.9005 | 0.3981 | 0.6335 | **0.8987** ⬅️ | **0.4053** | **0.6273** ⬅️ |
| XGBoostClassifier + EV | 0.9204 | 0.3184 | 0.7007 | **0.8967** | **0.4131** | **0.6219** |
| LogisticRegression + EV | 0.8974 | 0.4103 | 0.6246 | **0.8970** | **0.4122** | **0.6230** |
| LinearSVC + EV (calibrated) | 0.8928 | 0.4288 | 0.6039 | **0.8920** | **0.4321** | **0.6002** |

### Direct regression (round + clip)

| Regressor | Train Score | Train MAE | Train Acc | **Val Score** | **Val MAE** | **Val Acc** |
|---|---|---|---|---|---|---|
| MLPRegressor | 0.8989 | 0.4044 | 0.6247 | **0.8976** | **0.4096** | **0.6224** |
| XGBoostRegressor (MAE obj, retuned) | 0.9135 | 0.3459 | 0.6841 | **0.8930** | **0.4280** | **0.6129** |
| XGBoostRegressor (MAE obj, original) | 0.8974 | 0.4104 | 0.6279 | **0.8910** | **0.4360** | **0.6056** |
| Ridge | 0.8864 | 0.4542 | 0.5801 | **0.8868** | **0.4526** | **0.5811** |

## Takeaways

**Best Track 1 result so far:** MLPClassifier-v2 + expected-value decoding on `seq128` embeddings at **0.8987** validation score.

### Cross-variant comparison at matched classifier (argmax decoding)

| Classifier | default (128 tok, no prompt) | v2 (64 tok, prompt + norm) | seq128 (128 tok, prompt + norm) |
|---|---|---|---|
| MLPClassifier-v2 | 0.8950 | 0.8913 | **0.8974** |
| LogisticRegression | 0.8917 | 0.8890 | **0.8948** |
| LinearSVC | 0.8852 | 0.8857 | **0.8911** |
| RandomForest (RF-v2 on right two) | 0.8866 | 0.8852 | **0.8903** |
| KNeighborsClassifier | 0.8747 | 0.8813 | **0.8866** |

### Findings

1. **Sequence length matters.** Holding loading path / prompt / normalization constant (v2 vs seq128), increasing `max_seq_length` from 64 to 128 gave +0.0051 to +0.0058 across classifiers. Consistent with the data distribution: the 95th percentile of review length is ~98 words (~128 tokens), so seq64 was cutting off meaningful late-review content.

2. **Prompt + normalization has a subtler effect.** Default (plain loading, 128 tok) vs seq128 (prompt + norm, 128 tok) — same token budget, same model — seq128 outperforms default by +0.0024 (MLP) to +0.0059 (LinearSVC). Going further, even at seq64, prompt + norm helps distance-based classifiers (KNN: 0.8747 → 0.8813) more than linear ones.

3. **Expected-value decoding is a consistent free improvement.** Every classifier benefits from EV decoding over argmax, with no retraining required:
   - MLPClassifier-v2: 0.8974 → 0.8987 (+0.0013)
   - XGBoostClassifier: 0.8949 → 0.8967 (+0.0018)
   - LogisticRegression: 0.8948 → 0.8970 (+0.0022)
   - LinearSVC: 0.8911 → 0.8920 (+0.0009)
   
   The gain is larger for models with better-calibrated probability outputs (LR, XGBoost) and smaller for models with naturally peakier distributions (MLP) — consistent with what EV decoding actually does (uses the full probability distribution rather than just the mode).

4. **Direct MAE regression underperformed classification + EV decoding.** XGBoostRegressor with `objective="reg:absoluteerror"` reached 0.8930 after retuning, below XGBoostClassifier + EV at 0.8967. Cross-entropy gives the model a richer training signal (full distribution over classes) than L1 regression (single scalar target); combined with MAE-aware decoding at inference, classification + EV gets the best of both worlds. Only MLPRegressor came close to its classification+EV counterpart (0.8976 vs 0.8987).

5. **Ridge regression underperformed all other models.** 0.8868 val score — worse than every classifier variant. A linear regression on integer labels 0–4 produces a degenerate target distribution (bi/tri-modal around stars) that a linear model cannot capture well; linear classification treats the problem more appropriately.

### Methodological observations

**Unregularized neural nets overfit heavily.** `MLPClassifier` (256 hidden units, no regularization) reached 99.5% train accuracy but only 0.8593 val on default embeddings — severe overfit. `MLPClassifier-v2` (128 units, L2 α=1e-2, early stopping) fixed this and became the best classifier across all three variants.

**Unregularized random forests also overfit.** The plain `RandomForestClassifier` on default embeddings hit 100% train accuracy (memorization). `RandomForestClassifier-v2` with `max_depth=10, min_samples_leaf=5` was used on later variants.

**XGBoost training/validation gap decreases with softer objectives.** Argmax classification has gap 0.039, EV-decoded classification has 0.024, MAE regression has 0.006. Smaller gap is not necessarily better — the regression variant is simply underfitting compared to the classification+EV approach.